In [1]:
import os
os.chdir(r"..\models\..")

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pickle
from torch.utils.data import Dataset

from tokenizer import Tokenizer
from text_generator import TextGenerator

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [15]:
# Training Tokenizer
data = "data\\my_essays_data.txt"

with open(data, "r", encoding="utf-8") as f:
    text = f.read()

tokenizer = Tokenizer(vocab_size=100_000)
tokenizer.fit([text])

vocab_size = len(tokenizer.word_to_idx)
#print(f"Vocabulary Size: {vocab_size}")

with open("models/generator_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [3]:
class TextDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = seq_len // 2

        assert len(tokens) > seq_len, (
            f"Dataset too small! Tokens={len(tokens)}, seq_len={seq_len}"
        )

    def __len__(self):
        return (len(self.tokens) - self.seq_len) // self.stride
    
    def __getitem__(self, idx):

        start = idx * self.stride

        x = self.tokens[start:start + self.seq_len]
        y = self.tokens[start + 1:start + self.seq_len + 1]
       
        return torch.tensor(x), torch.tensor(y)

In [6]:
# Training the model
dataset = TextDataset(tokenizer.transform(text), seq_len=100)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextGenerator(
    vocab_size=vocab_size,
    embedding_dim=128,
    num_layers=3,
    num_heads=4,
    d_ff=512,
    max_len=128,
    dropout=0.1,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
batch_size = 64
epochs = 80

best_loss = float("inf")
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        output = model(x)

        loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
        loss.backward()

        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total_loss += loss.item()

    # Saving best model
    epoch_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "models/generator_best_model.pt")


Epoch 1, Loss: 8.0372
Epoch 2, Loss: 7.5199
Epoch 3, Loss: 7.2041
Epoch 4, Loss: 6.9643
Epoch 5, Loss: 6.7590
Epoch 6, Loss: 6.5925
Epoch 7, Loss: 6.4627
Epoch 8, Loss: 6.3665
Epoch 9, Loss: 6.2993
Epoch 10, Loss: 6.2513
Epoch 11, Loss: 6.2178
Epoch 12, Loss: 6.1886
Epoch 13, Loss: 6.1597
Epoch 14, Loss: 6.1291
Epoch 15, Loss: 6.0930
Epoch 16, Loss: 6.0546
Epoch 17, Loss: 6.0153
Epoch 18, Loss: 5.9724
Epoch 19, Loss: 5.9283
Epoch 20, Loss: 5.8858
Epoch 21, Loss: 5.8394
Epoch 22, Loss: 5.7966
Epoch 23, Loss: 5.7524
Epoch 24, Loss: 5.7099
Epoch 25, Loss: 5.6661
Epoch 26, Loss: 5.6212
Epoch 27, Loss: 5.5764
Epoch 28, Loss: 5.5350
Epoch 29, Loss: 5.4919
Epoch 30, Loss: 5.4499
Epoch 31, Loss: 5.4064
Epoch 32, Loss: 5.3623
Epoch 33, Loss: 5.3190
Epoch 34, Loss: 5.2790
Epoch 35, Loss: 5.2369
Epoch 36, Loss: 5.1940
Epoch 37, Loss: 5.1522
Epoch 38, Loss: 5.1115
Epoch 39, Loss: 5.0773
Epoch 40, Loss: 5.0334
Epoch 41, Loss: 4.9936
Epoch 42, Loss: 4.9492
Epoch 43, Loss: 4.9163
Epoch 44, Loss: 4.87

In [26]:
# Trying model

with open("models/generator_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextGenerator(
    vocab_size=vocab_size,
    embedding_dim=128,
    num_layers=3,
    num_heads=4,
    d_ff=512,
    max_len=128,
    dropout=0.1,
    device=device
).to(device)

model.load_state_dict(torch.load("models/generator_best_model.pt", map_location=device))

model.eval()

def generation_text(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()

    tokens = tokenizer.transform(prompt)
    tokens = torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(tokens)

        next_token_logits = logits[:, -1, :]
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=-1)

    generated = tokens.squeeze().tolist()

    words = [tokenizer.idx_to_word.get(t, "<UNK>") for t in generated]
    lines = [" ".join(words[i:i+10]) for i in range(0, len(words), 10)]

    return "\n".join(lines)

print("Text generation to my prompt:\n")
print(generation_text(
    model,
    tokenizer,
    prompt="I was very happy because",
    max_new_tokens=30
))

Text generation to my prompt:

i was very happy because and led to make the
idea of me easy that are the majority , government
to be viewed as a lot of these obligation conclude
had contrasting from because it
